# Event logs

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LukasLiss/totem-tool/blob/main/totem_lib/docs/guide/event_logs.ipynb)

totem-lib reads object-centric event logs in the OCEL 2.0 format (Berti et al., 2023). Such a log has events and objects. Each event has an activity and a timestamp and touches one or more objects. Each object has a type, such as `order` or `item`. This notebook loads a tiny log in the two forms that totem-lib uses, queries it with SQL, filters it, and adds a column to its events.

## Setup

In Google Colab, the first cell installs totem-lib from PyPI, where it is called `totem-tool`. For normal usage, install totem-lib first as shown in [Installation](../installation.md).

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install --quiet totem-tool "pulp<4"

The example log comes from the totem-lib repository. Outside a clone of the repository, the next cell downloads it (11 KB).

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

LOG = Path("../../test_data/small/resource_aware_orders.json")
if not LOG.exists():
    LOG = Path("resource_aware_orders.json")
    if not LOG.exists():
        urlretrieve(
            "https://raw.githubusercontent.com/LukasLiss/totem-tool/main/"
            "totem_lib/test_data/small/resource_aware_orders.json",
            LOG,
        )

## Two ways to load a log

totem-lib has two forms of an event log:

- `import_ocel` returns an `ObjectCentricEventLog`. It keeps the log in [Polars](https://pola.rs) DataFrames in memory.
- `import_ocel_db` returns an `OcelDuckDB`. It keeps the log in a [DuckDB](https://duckdb.org) database, in memory or in a file. This form also works for large logs, and you can query it with SQL.

Both read the formats of OCEL 2.0 and pick the format from the file extension: `.sqlite`, `.json`, `.xml` or `.csv`. They also open `.duckdb` files that totem-lib saved before.

Some functions take one form, some the other, and some take both. For example, `totemDiscovery` takes an `ObjectCentricEventLog`, while `find_variants`, `conformance_of_totem` and the filters below take an `OcelDuckDB`. The [API reference](../api/index.rst) says which form each function takes. `OcelDuckDB(ocel)` turns an `ObjectCentricEventLog` into a database.

## ObjectCentricEventLog

The log has four orders with six items, packed into three packages. One worker, `w1`, picks, packs and ships everything.

In [ ]:
import polars as pl
from totem_lib import import_ocel

pl.Config.set_tbl_rows(25)  # show every row of these small tables
ocel = import_ocel(LOG)
ocel.events

Each event has an id, an activity, a timestamp in Unix seconds, and the list of objects it touches. Each object has a qualifier that says its role in the event. The objects table has the type of each object and its object-to-object relations, also with qualifiers:

In [ ]:
ocel.objects

A few methods answer common questions about events and objects:

In [ ]:
print(sorted(ocel.object_types))
print(ocel.get_event_activity("e06"))
print(ocel.get_event_objects_by_type("e06", "item"))

## OcelDuckDB

`import_ocel_db` reads the log into a DuckDB database. Without `db_path`, the database stays in memory.

In [ ]:
from totem_lib.ocel import import_ocel_db

db = import_ocel_db(LOG)
db.query("SHOW TABLES")

The database has five tables:

- `events`: `event_id`, `activity`, `timestamp_unix` and one column per event attribute
- `objects`: `obj_id`, `obj_type` and one column per object attribute
- `event_object`: which event touches which object, with the qualifier
- `object_relations`: the object-to-object relations, with the qualifier
- `object_attribute_history`: the values of the object attributes over time

All attribute columns hold text. Cast them in SQL when you need numbers, for example `price::DOUBLE`. This log has no attributes.

`query` runs SQL and returns a Polars DataFrame. This query counts the events per object type:

In [ ]:
db.query(
    """
    SELECT o.obj_type, COUNT(DISTINCT eo.event_id) AS events
    FROM objects o
    JOIN event_object eo ON o.obj_id = eo.obj_id
    GROUP BY o.obj_type
    ORDER BY events DESC
    """
)

Reading a large log takes time. To read it only once, save the database to a file and load the file next time. Opening a saved file is fast. `import_ocel_db(LOG, db_path="log.duckdb")` writes the file right away.

In [ ]:
import tempfile
from totem_lib.ocel import OcelDuckDB

DB_FILE = Path(tempfile.mkdtemp()) / "resource_aware_orders.duckdb"
db.save(str(DB_FILE))

saved = OcelDuckDB.load(str(DB_FILE), read_only=True)
print(saved.query("SELECT COUNT(*) AS events FROM events"))
saved.close()

With `read_only=True`, several programs can open the same file at once.

## Filter a log

`apply_filter_stack` filters an `OcelDuckDB`. It returns a new database in memory and some statistics. The original database does not change.

A filter stack is a list of rules. Each rule has an `id`, a `type` and `params`:

- `activity`: keep the events with one of the activities in `include`
- `object_types`: keep the objects with one of the types in `include`
- `time_range`: keep the events between `after` and `before`, in Unix seconds. Both bounds are optional and included.

An event or object has to pass every rule. In the end, events without objects and objects without events are removed.

This filter keeps only the orders and items:

In [ ]:
from totem_lib import apply_filter_stack

orders_and_items, stats = apply_filter_stack(
    db,
    {"filters": [{"id": "orders-and-items", "type": "object_types", "params": {"include": ["order", "item"]}}]},
)
stats

The worker and the packages are gone. So are the events that touched only them, such as `ship package` and the two shift events. The other events keep only their orders and items:

In [ ]:
orders_and_items.query(
    "SELECT activity, COUNT(*) AS events FROM events GROUP BY activity ORDER BY activity"
)

Rules of different types work together. This filter keeps the picks and packs until 18:00 on the first day:

In [ ]:
from datetime import datetime, timezone

six_pm = int(datetime(2026, 3, 2, 18, tzinfo=timezone.utc).timestamp())
early_work, stats = apply_filter_stack(
    db,
    {
        "filters": [
            {"id": "until-6pm", "type": "time_range", "params": {"before": six_pm}},
            {"id": "work", "type": "activity", "params": {"include": ["pick item", "pack items"]}},
        ]
    },
)
early_work.get_events_with_objects().group_by("event_id", "activity", maintain_order=True).agg(
    objects=pl.col("obj_id").sort().str.join(", ")
)

`FilterStack` and `FilterRule` build the same stack as classes. A rule with `enabled=False` is skipped. With `stats_only=True`, `apply_filter_stack` only counts and does not build a new database:

In [ ]:
from totem_lib import FilterRule, FilterStack

stack = FilterStack(
    [
        FilterRule(id="picks", type="activity", enabled=True, params={"include": ["pick item"]}),
        FilterRule(id="items", type="object_types", enabled=False, params={"include": ["item"]}),
    ]
)
_, stats = apply_filter_stack(db, stack, stats_only=True)
print(f"{stats['event_count_after']} of {stats['event_count_before']} events")

For an `ObjectCentricEventLog`, `filter_dead_objects` removes objects that no event touches. Such objects can appear in inconsistent log files.

## Event columns

The `events` table can hold your own columns. Use them to store one value per event, for example the process execution that each event belongs to. Later steps can read the column again: the [OCCN replay](occn_replay_fitness.md) can replay exactly the executions stored in a column.

The next cell splits the log into process executions, as in [Variants](variants.ipynb), and stores the execution of each event in a new column `execution`. `write_event_column` creates the column if needed and returns the number of events that got a value.

In [ ]:
from totem_lib import extract_process_executions, write_event_column

executions = extract_process_executions(
    db, extraction="resource_aware", business_object_types=["order", "item", "package"]
)
execution_of_event = {
    event_id: case_id
    for case_id, event_ids in executions.case_events.items()
    for event_id in event_ids
}
write_event_column(db.conn, "execution", execution_of_event)

The two shift events belong to no execution, so their value stays empty (`null`). Now the column is part of the database:

In [ ]:
from totem_lib import event_column_summary, list_event_columns

print(list_event_columns(db.conn))
print(event_column_summary(db.conn, "execution"))
db.query(
    "SELECT execution, COUNT(*) AS events FROM events GROUP BY execution ORDER BY execution"
)

Writing a column again replaces all its old values. Column names have 1 to 64 characters, and `event_id`, `activity` and `timestamp_unix` cannot be used. `validate_event_column_name` checks a name and raises an `EventColumnError` if it is not allowed.

`write_event_columns_to_file` writes columns into a saved database file. It opens the file, writes the columns and closes the file again. No other connection may have the file open at that time.

In [ ]:
from totem_lib import write_event_columns_to_file

write_event_columns_to_file(str(DB_FILE), {"execution": execution_of_event})

## Next steps

- [TOTeM models](totem.ipynb) and [Variants](variants.ipynb) work on logs loaded as shown here.
- The [API reference](../api/event_logs.rst) lists all functions for event logs.

## References

- Berti, A., Koren, I., Adams, J.N., Park, G., Knopp, B., Graves, N., Rafiei, M., Liß, L., Unterberg, L.T.G., Zhang, Y. (2023). OCEL (Object-Centric Event Log) 2.0 Specification. <https://www.ocel-standard.org>